In [2]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough

llm = ChatOllama(base_url="http://localhost:11434", model="llama3.2:1b")

#### 1. Sequential Chain

In [ ]:
# First chain: generate facts
fact_prompt = ChatPromptTemplate.from_template(
    '''
    Tell me {number} facts about {topic} in short bullet points.
    '''
)
fact_chain = fact_prompt | llm | StrOutputParser()


# Second chain: analyze complexity
analysis_prompt = ChatPromptTemplate.from_template(
    '''
    Analyze the complexity level (school, college, expert) of this text: {text}. 
    Answer in one sentence.
    '''
)
analysis_chain = analysis_prompt | llm | StrOutputParser()

# Getting facts and analysis together
sequential_chain = RunnableParallel(
    facts=fact_chain,
    analysis={"text": fact_chain} | analysis_chain
)

result = sequential_chain.invoke({"number": 2, "topic": "cosmic dawns"})

print("Facts:")
print(result["facts"])
print()
print("Analysis:")
print(result["analysis"])

Facts:
Here are two facts about cosmic dawns:

• Cosmic dawns occur when the Earth passes through a region of extremely high solar radiation, often due to intense aurorae or intense magnetic storms.
• During these events, the atmosphere can be illuminated by the intense light of charged particles from the solar wind and coronal mass ejections, creating spectacular displays known as "cosmic dawns".

Analysis:
The complexity level of this text is likely "school", as it consists of short bullet points and basic definitions without any advanced concepts or technical jargon.


#### 2. Parallel Chains

In [ ]:
fact_chain_parallel = (
    ChatPromptTemplate.from_template("Tell me 2 facts about {topic}")
    | llm
    | StrOutputParser()
)

poem_chain = (
    ChatPromptTemplate.from_template("Write a 2-line poem about {topic}")
    | llm
    | StrOutputParser()
)

parallel_chain = RunnableParallel(
    facts=fact_chain_parallel,
    poem=poem_chain
)
result = parallel_chain.invoke({"topic": "stars"})
print("Facts:", result["facts"])
print("Poem:", result["poem"])

Facts: Here are two facts about stars:

1. Stars are massive balls of hot, glowing gas. They are primarily composed of hydrogen and helium, which are formed in the star's core through nuclear reactions that occur when hydrogen atoms are fused together to form helium.

2. The largest star known to date is VY Canis Majoris, located in the constellation Canis Major. It has a radius more than 2,100 times larger than our sun and is estimated to be around 5,000 light-years away from Earth.
Poem: Here is a 2-line poem about stars:

Twinkling lights in the midnight sky,
A celestial show, beyond our sigh.


#### 3. Routing Chain

In [9]:
# Classifier: Positive or Negative Review
classifier_prompt = ChatPromptTemplate.from_template("Classify this review as 'positive' or 'negative'. Print only one word: {text}")
classifier_chain = classifier_prompt | llm | StrOutputParser()

positive_handler = (
    ChatPromptTemplate.from_template(
        """
        The user gave a positive review: {text}
        
        Acknowledge their feedback, thank them genuinely, and let them know you appreciate their support.
        Keep it warm and professional. Reply in 2-3 sentences.
        """
    ) | llm  | StrOutputParser()
)

negative_handler = (
    ChatPromptTemplate.from_template(
        """
        The user gave a negative review: {text}
        
        Acknowledge their concern, apologize sincerely, and let them know their feedback will be taken seriously.
        Keep it empathetic and professional. Reply in 2-3 sentences.
        """
    ) | llm | StrOutputParser()
)

def route(info):
    return positive_handler if "positive" in info["sentiment"].lower() else negative_handler

routing_chain = (
    {"sentiment": classifier_chain, "text": RunnablePassthrough()} | RunnableLambda(route)
)

# Testing with a random review
review = "The product is amazing! Works exactly as described. Highly recommended."
result = routing_chain.invoke(review)
print("Review:", review)
print("Response:", result)

Review: The product is amazing! Works exactly as described. Highly recommended.
Response: Here's a possible response:

"Dear valued customer, we're thrilled to hear that you've enjoyed our product! We're grateful for your loyalty and support - your feedback means the world to us. Thank you for choosing our company and for taking the time to share your positive experience with others."
